In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torchvision

In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)) # MNIST standard mean and std
])
train_dataset = torchvision.datasets.MNIST(root='./data_alex', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data_alex', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

100.0%
100.0%
100.0%
100.0%


In [2]:
class VGG(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=64, kernel_size=(3,3), stride=1, padding=1)
        self.conv2 = nn.Conv2d(in_channels=64, out_channels=64,kernel_size=(3,3), stride=1, padding=1)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128,kernel_size=(3,3), stride=1, padding=1)
        self.conv4 = nn.Conv2d(in_channels=128, out_channels=128,kernel_size=(3,3), stride=1, padding=1)
        self.conv5 = nn.Conv2d(in_channels=128, out_channels=256,kernel_size=(3,3), stride=1, padding=1)
        self.conv6 = nn.Conv2d(in_channels=256, out_channels=256,kernel_size=(3,3), stride=1, padding=1)
        self.conv7 = nn.Conv2d(in_channels=256, out_channels=256,kernel_size=(3,3), stride=1, padding=1)
        self.conv8 = nn.Conv2d(in_channels=256, out_channels=512,kernel_size=(3,3), stride=1, padding=1)
        self.conv9 = nn.Conv2d(in_channels=512, out_channels=512,kernel_size=(3,3), stride=1, padding=1)
        self.conv10 = nn.Conv2d(in_channels=512, out_channels=512,kernel_size=(3,3), stride=1, padding=1)
        self.conv11 = nn.Conv2d(in_channels=512, out_channels=512,kernel_size=(3,3), stride=1, padding=1)
        self.conv12 = nn.Conv2d(in_channels=512, out_channels=512,kernel_size=(3,3), stride=1, padding=1)
        self.conv13 = nn.Conv2d(in_channels=512, out_channels=512,kernel_size=(3,3), stride=1, padding=1)
        self.max_pool = nn.MaxPool2d(2,2)
        self.dropout = nn.Dropout(p=0.5)
        self.fc1 = nn.Linear(25088,4096)
        self.fc2 = nn.Linear(4096,4096)
        self.fc3 = nn.Linear(4096,10)

    def forward(self,x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = self.max_pool(x)
        x = self.conv3(x)
        x = F.relu(x)
        x = self.conv4(x)
        x = F.relu(x)
        x = self.max_pool(x)
        x = self.conv5(x)
        x = F.relu(x)
        x = self.conv6(x)
        x = F.relu(x)
        x = self.conv7(x)
        x = F.relu(x)
        x = self.max_pool(x)
        x = self.conv8(x)
        x = F.relu(x)
        x = self.conv9(x)
        x = F.relu(x)
        x = self.conv10(x)
        x = F.relu(x)
        x = self.max_pool(x)
        x = self.conv11(x)
        x = F.relu(x)
        x = self.conv12(x)
        x = F.relu(x)
        x = self.conv13(x)
        x = F.relu(x)
        x = self.max_pool(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc3(x)
        return x

model = VGG()
device ='cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

In [ ]:
# Defining Loss Function (Since it is a multi-class classification, I chose CrossEntropyLoss)
loss_fn = nn.CrossEntropyLoss()
# Defining Optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

In [ ]:
# Setup optimization loop(s)
epochs = 100

### Train time
# Loop through the epochs
for epoch in range(epochs):
    train_loss_total = 0
    test_loss_total = 0
    train_correct_guess_total = 0
    test_correct_guess_total = 0
    # Set the model to train mode (this is the default)
    model.train(True)
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        # 1. Do the forward pass
        y_pred = model(images)
        predicted_labels = y_pred.argmax(dim=1)
        train_correct_guess_total += (predicted_labels == labels).sum().item()
        # 2. Calculate the loss (how wrong the model is)
        loss = loss_fn(y_pred, labels)
        # 3. Zero the optimizer gradients
        optimizer.zero_grad()
        # 4. Perform backpropagation
        loss.backward()
        # 5. Step the optimizer
        optimizer.step()
        train_loss_total += loss.item() * images.size(0)
    training_accuracy = (train_correct_guess_total / len(train_dataset)) * 100
    train_loss_total /= len(train_dataset)
    ### Test time
    # Set the model to eval mode
    model.eval()
    # Turn on inference mode context manager
    with torch.inference_mode():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            # 1. Do the forward pass
            test_pred = model(images)
            predicted_labels = test_pred.argmax(dim=1)
            test_correct_guess_total += (predicted_labels == labels).sum().item()
            # 2. Calculate the loss
            test_loss = loss_fn(test_pred, labels)
            test_loss_total += test_loss.item() * images.size(0)
    test_accuracy = (test_correct_guess_total / len(test_dataset)) * 100
    test_loss_total /= len(test_dataset)

    # Print out what's happening
    print(f"Epoch: {epoch + 1} | Train loss: {train_loss_total:.4f} | Test loss: {test_loss_total:.4f} | Training Accuracy: {training_accuracy:.2f} % | Test Accuracy: {test_accuracy:.2f} % ")

In [ ]:
torch.save(model.state_dict(), 'vgg.pth')